# Act 4 - Multi-Variant Text Analysis and Generation (BERT, GPT, and GANs)

---

**Submitted by:**

| Name | Role |
|------|------|
| Cyprian Andrew Garpida | Student |
| Dharl Russell Perez | Student |
| Jastine Marie Sabado | Student |


Installs all required Python libraries for NLP tasks


In [ ]:
!pip install -q transformers datasets evaluate accelerate nltk rouge-score scikit-learn torch

## Section 1: Environment Setup & Dataset Download

> Imports all required libraries, sets random seeds for reproducibility,
> detects available device (GPU/CPU), and downloads the sentiment analysis dataset from Google Drive.

In [ ]:
import os, re, time, random
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForCausalLM, DataCollatorWithPadding, DataCollatorForLanguageModeling,
    TrainingArguments, Trainer
)
import evaluate

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import gdown

FILE_ID = "168Wiqn6zwQWwibrYS_dNs3QWFnI7KyQu"
DATA_PATH = "sentiment_analysis_dataset.csv"
gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", DATA_PATH, quiet=False)

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=168Wiqn6zwQWwibrYS_dNs3QWFnI7KyQu
To: /content/sentiment_analysis_dataset.csv
100%|██████████| 396k/396k [00:00<00:00, 53.1MB/s]


'sentiment_analysis_dataset.csv'

### Loads, cleans, and splits the dataset
 reads the CSV, removes nulls, strips URLs and extra whitespace, converts sentiment labels to binary (Positive = 1), then splits the data into 70% train / 15% validation / 15% test sets and saves each split as a CSV file.

In [ ]:
df = pd.read_csv(DATA_PATH, engine="python", on_bad_lines="skip")
df = df.dropna(subset=["Comment", "Sentiment"]).reset_index(drop=True)

def clean_text(t):
    t = str(t).strip()
    t = re.sub(r"http\S+|www\.\S+", "", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

df["Comment"] = df["Comment"].apply(clean_text)
df = df[df["Comment"].str.len() > 0].reset_index(drop=True)
df["label"] = (df["Sentiment"] == "Positive").astype(int)
print(df["Sentiment"].value_counts())
print(df["label"].value_counts(normalize=True))

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"])
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

os.makedirs("splits", exist_ok=True)
train_df.to_csv("splits/train.csv", index=False)
val_df.to_csv("splits/val.csv", index=False)
test_df.to_csv("splits/test.csv", index=False)

Sentiment
Positive    4679
Neutral     1976
Negative      79
Name: count, dtype: int64
label
1    0.694832
0    0.305168
Name: proportion, dtype: float64
Train: (4713, 3) Val: (1010, 3) Test: (1011, 3)


### Fine-tunes BERT for sentiment classification
 >loads the pre-trained `bert-base-uncased` model, tokenizes the dataset, trains it for 3 epochs with evaluation per epoch, measures accuracy/precision/recall/F1, prints a full classification report on the test set, then saves the fine-tuned model locally.

In [ ]:
BERT_MODEL_NAME = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

def to_hf_dataset(df_):
    return Dataset.from_pandas(df_[["Comment", "label"]].rename(columns={"Comment": "text"}))

def tokenize_fn(batch):
    return bert_tokenizer(batch["text"], truncation=True, max_length=128)

train_ds, val_ds, test_ds = [to_hf_dataset(d).map(tokenize_fn, batched=True) for d in (train_df, val_df, test_df)]
data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=2).to(device)
precision_metric, recall_metric, f1_metric, accuracy_metric = [
    evaluate.load(m) for m in ("precision", "recall", "f1", "accuracy")
]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

bert_training_args = TrainingArguments(
    output_dir="bert_out", eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=3, weight_decay=0.01, load_best_model_at_end=True,
    metric_for_best_model="f1", logging_steps=50, report_to="none",
)

bert_trainer = Trainer(
    model=bert_model, args=bert_training_args, train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=bert_tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
)

bert_start = time.time()
bert_trainer.train()
bert_time_per_epoch = (time.time() - bert_start) / bert_training_args.num_train_epochs
print(f"BERT time/epoch: {bert_time_per_epoch:.1f}s")

bert_test_metrics = bert_trainer.evaluate(test_ds)
test_preds = bert_trainer.predict(test_ds)
y_pred = np.argmax(test_preds.predictions, axis=-1)
print(classification_report(test_preds.label_ids, y_pred, target_names=["Not-Positive", "Positive"]))

bert_model.save_pretrained("bert_finetuned")
bert_tokenizer.save_pretrained("bert_finetuned")
print(bert_test_metrics)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

BERT time/epoch: 48.6s


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.011559,0.080340,3,0.987141,0.985896,0.995726,0.990787


              precision    recall  f1-score   support

Not-Positive       0.99      0.97      0.98       309
    Positive       0.99      1.00      0.99       702

    accuracy                           0.99      1011
   macro avg       0.99      0.98      0.98      1011
weighted avg       0.99      0.99      0.99      1011



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.08034004271030426, 'eval_accuracy': 0.9871414441147379, 'eval_precision': 0.9858956276445698, 'eval_recall': 0.9957264957264957, 'eval_f1': 0.9907866761162296}


### Fine-tunes DistilGPT-2 for text generation
> Loads `distilgpt2`, trains it as a causal language model on the comments dataset for 3 epochs, evaluates perplexity, generates sample continuations from test prompts, scores outputs using BLEU and ROUGE metrics, then saves the fine-tuned model locally.

In [ ]:
GPT_MODEL_NAME = "distilgpt2"
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL_NAME)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

def gpt_tokenize_fn(batch):
    return gpt_tokenizer(batch["text"], truncation=True, max_length=64)

gpt_train_ds, gpt_val_ds, gpt_test_ds = [
    Dataset.from_pandas(d[["Comment"]].rename(columns={"Comment": "text"})).map(
        gpt_tokenize_fn, batched=True, remove_columns=["text"])
    for d in (train_df, val_df, test_df)
]
gpt_collator = DataCollatorForLanguageModeling(tokenizer=gpt_tokenizer, mlm=False)

gpt_model = AutoModelForCausalLM.from_pretrained(GPT_MODEL_NAME).to(device)
gpt_training_args = TrainingArguments(
    output_dir="gpt_out", eval_strategy="epoch", save_strategy="epoch",
    learning_rate=5e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=3, weight_decay=0.01, logging_steps=50, report_to="none",
)
gpt_trainer = Trainer(model=gpt_model, args=gpt_training_args, train_dataset=gpt_train_ds,
                       eval_dataset=gpt_val_ds, data_collator=gpt_collator)

gpt_start = time.time()
gpt_trainer.train()
gpt_time_per_epoch = (time.time() - gpt_start) / gpt_training_args.num_train_epochs

gpt_eval_metrics = gpt_trainer.evaluate(gpt_test_ds)
gpt_perplexity = float(np.exp(gpt_eval_metrics["eval_loss"]))
print(f"GPT time/epoch: {gpt_time_per_epoch:.1f}s | perplexity: {gpt_perplexity:.2f}")

gpt_model.eval()
def generate_continuation(prompt, max_new_tokens=30):
    inputs = gpt_tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = gpt_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=True,
        top_k=50, top_p=0.95, temperature=0.9, pad_token_id=gpt_tokenizer.eos_token_id,
    )
    return gpt_tokenizer.decode(output_ids[0], skip_special_tokens=True)

sample_prompts = ["This video is", "I think the content creator", "Honestly the reaction to this is"]
generated_samples = [generate_continuation(p) for p in sample_prompts]
for p, g in zip(sample_prompts, generated_samples):
    print(f"PROMPT: {p!r}\nGENERATED: {g!r}\n")

bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
references = test_df["Comment"].tolist()[:50]
generated_for_eval = [
    generate_continuation(" ".join(r.split()[:max(1, len(r.split())//3)]), max_new_tokens=20)
    for r in references
]
bleu_score = bleu_metric.compute(predictions=generated_for_eval, references=[[r] for r in references])
rouge_score = rouge_metric.compute(predictions=generated_for_eval, references=references)
print("GPT-2 BLEU:", bleu_score["bleu"], "| ROUGE:", rouge_score)

gpt_model.save_pretrained("gpt2_finetuned")
gpt_tokenizer.save_pretrained("gpt2_finetuned")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Training Loss,Validation Loss,Epoch
2.502343,2.610385,3


GPT time/epoch: 41.8s | perplexity: 13.60
PROMPT: 'This video is'
GENERATED: "This video is the best example of this, it shows what's possible for a coach to survive without emotional stress. Absolute perfection. This video proves that. Absolute perfection"

PROMPT: 'I think the content creator'
GENERATED: "I think the content creator is doing all of this to help the world better, but there's so much money in the making. Thanks! Anytime someone is spending that kind"

PROMPT: 'Honestly the reaction to this is'
GENERATED: 'Honestly the reaction to this is INSANELY emotional. It truly shows me just how much of a blessing the coach gave me. Thank you for helping so many people out there.'



GPT-2 BLEU: 0.08681267690234765 | ROUGE: {'rouge1': np.float64(0.238349364772731), 'rouge2': np.float64(0.13056237671725177), 'rougeL': np.float64(0.23491236875098648), 'rougeLsum': np.float64(0.23557703256620255)}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('gpt2_finetuned/tokenizer_config.json', 'gpt2_finetuned/tokenizer.json')

### Builds the GAN architecture for text generation
> Constructs a custom vocabulary from training comments, encodes all text into fixed-length token sequences, then defines a Generator (LSTM-based, uses Gumbel-Softmax for differentiable token sampling) and a Discriminator (CNN-based) for adversarial text generation.

In [ ]:
MAX_LEN = 20
MIN_FREQ = 2

def simple_tokenize(text):
    return re.findall(r"[a-zA-Z']+", text.lower())

all_tokens = [tok for c in train_df["Comment"] for tok in simple_tokenize(c)]
counts = Counter(all_tokens)
vocab = ["<PAD>", "<UNK>", "<EOS>"] + [w for w, c in counts.items() if c >= MIN_FREQ]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab)
print("Vocab size:", VOCAB_SIZE)

def encode(text):
    toks = simple_tokenize(text)[: MAX_LEN - 1]
    ids = [word2idx.get(t, word2idx["<UNK>"]) for t in toks] + [word2idx["<EOS>"]]
    ids = ids + [word2idx["<PAD>"]] * (MAX_LEN - len(ids))
    return ids[:MAX_LEN]

train_encoded = torch.tensor([encode(c) for c in train_df["Comment"]], dtype=torch.long)
print("Encoded training tensor shape:", train_encoded.shape)

class Generator(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden_dim=128, max_len=MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.noise_to_hidden = nn.Linear(100, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, z, temperature=1.0):
        batch_size = z.size(0)
        h0 = torch.tanh(self.noise_to_hidden(z)).unsqueeze(0)
        c0 = torch.zeros_like(h0)
        input_tok = torch.zeros(batch_size, 1, dtype=torch.long, device=z.device)
        hidden = (h0, c0)
        outputs = []
        inp_emb = self.embedding(input_tok)
        for t in range(self.max_len):
            out, hidden = self.lstm(inp_emb, hidden)
            logits = self.out(out.squeeze(1))
            gumbel = F.gumbel_softmax(logits, tau=temperature, hard=False)
            outputs.append(gumbel.unsqueeze(1))
            inp_emb = (gumbel @ self.embedding.weight).unsqueeze(1)
        return torch.cat(outputs, dim=1)

    def sample_ids(self, z, temperature=0.8):
        with torch.no_grad():
            return self.forward(z, temperature=temperature).argmax(dim=-1)

class Discriminator(nn.Module):
    def __init__(self, vocab_size, emb_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.conv1 = nn.Conv1d(emb_dim, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        emb = (x @ self.embedding.weight) if x.dim() == 3 else self.embedding(x)
        emb = emb.transpose(1, 2)
        h = F.leaky_relu(self.conv1(emb), 0.2)
        h = F.leaky_relu(self.conv2(h), 0.2)
        return self.fc(self.pool(h).squeeze(-1))

def to_one_hot(ids, vocab_size):
    return F.one_hot(ids, num_classes=vocab_size).float()

def ids_to_text(ids_row):
    words = []
    for i in ids_row.tolist():
        w = idx2word.get(i, "<UNK>")
        if w == "<EOS>": break
        if w != "<PAD>": words.append(w)
    return " ".join(words)

Vocab size: 1871
Encoded training tensor shape: torch.Size([4713, 20])


### Trains the GAN and evaluates generated text
> Initializes the Generator and Discriminator with Adam optimizers, runs adversarial training for 30 epochs (Discriminator learns to spot fakes, Generator learns to fool it), samples generated sentences, then evaluates the Discriminator's accuracy/F1 and scores generated text quality using BLEU and ROUGE metrics.

In [ ]:
NOISE_DIM = 100
generator = Generator(VOCAB_SIZE).to(device)
discriminator = Discriminator(VOCAB_SIZE).to(device)
g_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.5, 0.999))
bce_loss = nn.BCEWithLogitsLoss()

def real_batch(batch_size):
    idx = torch.randint(0, train_encoded.size(0), (batch_size,))
    return train_encoded[idx].to(device)

GAN_EPOCHS, BATCH_SIZE = 30, 64
gan_epoch_times, d_loss_history, g_loss_history = [], [], []

for epoch in range(GAN_EPOCHS):
    t0 = time.time()
    discriminator.train(); generator.train()
    real_one_hot = to_one_hot(real_batch(BATCH_SIZE), VOCAB_SIZE)
    z = torch.randn(BATCH_SIZE, NOISE_DIM, device=device)
    fake_soft = generator(z).detach()

    d_optimizer.zero_grad()
    d_loss = bce_loss(discriminator(real_one_hot), torch.ones(BATCH_SIZE, 1, device=device) * 0.9) + \
             bce_loss(discriminator(fake_soft), torch.zeros(BATCH_SIZE, 1, device=device))
    d_loss.backward(); d_optimizer.step()

    z = torch.randn(BATCH_SIZE, NOISE_DIM, device=device)
    g_optimizer.zero_grad()
    g_loss = bce_loss(discriminator(generator(z)), torch.ones(BATCH_SIZE, 1, device=device))
    g_loss.backward(); g_optimizer.step()

    gan_epoch_times.append(time.time() - t0)
    d_loss_history.append(d_loss.item()); g_loss_history.append(g_loss.item())
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{GAN_EPOCHS} | D: {d_loss.item():.4f} | G: {g_loss.item():.4f}")

gan_time_per_epoch = float(np.mean(gan_epoch_times))

generator.eval(); discriminator.eval()
z = torch.randn(8, NOISE_DIM, device=device)
gan_samples = [ids_to_text(row) for row in generator.sample_ids(z, 0.8)]
print("Sample GAN sentences:", gan_samples)

n_eval = 200
real_eval_oh = to_one_hot(train_encoded[torch.randint(0, train_encoded.size(0), (n_eval,))].to(device), VOCAB_SIZE)
z_eval = torch.randn(n_eval, NOISE_DIM, device=device)
with torch.no_grad():
    real_preds = (torch.sigmoid(discriminator(real_eval_oh)) > 0.5).long().cpu().numpy().flatten()
    fake_preds = (torch.sigmoid(discriminator(generator(z_eval))) > 0.5).long().cpu().numpy().flatten()
y_true_disc = np.concatenate([np.ones(n_eval), np.zeros(n_eval)])
y_pred_disc = np.concatenate([real_preds, fake_preds])
disc_acc = accuracy_score(y_true_disc, y_pred_disc)
disc_precision = precision_score(y_true_disc, y_pred_disc, zero_division=0)
disc_recall = recall_score(y_true_disc, y_pred_disc, zero_division=0)
disc_f1 = f1_score(y_true_disc, y_pred_disc, zero_division=0)
print(f"Discriminator Acc={disc_acc:.3f} P={disc_precision:.3f} R={disc_recall:.3f} F1={disc_f1:.3f}")

n_gen_eval = 50
gan_generated_texts = [ids_to_text(r) or "<EMPTY>" for r in generator.sample_ids(torch.randn(n_gen_eval, NOISE_DIM, device=device), 0.8)]
gan_references = test_df["Comment"].tolist()[:n_gen_eval]
gan_bleu = bleu_metric.compute(predictions=gan_generated_texts, references=[[r] for r in gan_references])
gan_rouge = rouge_metric.compute(predictions=gan_generated_texts, references=gan_references)
print("Text-GAN BLEU:", gan_bleu["bleu"], "| ROUGE:", gan_rouge)

Epoch 5/30 | D: 1.3301 | G: 0.7582
Epoch 10/30 | D: 1.2817 | G: 0.7528
Epoch 15/30 | D: 1.2297 | G: 0.7481
Epoch 20/30 | D: 1.1982 | G: 0.7408
Epoch 25/30 | D: 1.1594 | G: 0.7425
Epoch 30/30 | D: 1.1170 | G: 0.7397
Sample GAN sentences: ['think transformation mean willing version brooooo chills proof officer winning colombia video lucia astronaut friend total hand possible no wanna', "foul coach peace thats yacht friends were woo can't jobs constantly heartbreaking joke hail lock having colombia fuels club group", "glad carbon ton gaza worry influential very endlessly stealing insanely 'i ittt brian smaller wanted accomplished cause worry truck humanity", 'location smoking effortlessly cockpit mac pays hour elenmi eragon paper shampoo progress hersheys but sentence friendship inspired has edit honored', 'achievement arc turning obsession beauty been prize stepped trigonometry yard champion rewatched off complete teamwork by absolutely teared smile kiss', "curious prize training nuclear

### Compares all models and saves results to Google Drive
> Builds a summary table comparing BERT, GPT-2, and GAN across precision/recall/F1, BLEU/ROUGE/perplexity, and training speed, saves the comparison CSV and generated text samples locally under `./results`, then

In [ ]:
comparison_df = pd.DataFrame({
    "Model Variant": ["BERT", "GPT-2 (DistilGPT-2)", "Text-GAN"],
    "Primary Metric (P/R/F1)": [
        f"P={bert_test_metrics['eval_precision']:.3f} R={bert_test_metrics['eval_recall']:.3f} F1={bert_test_metrics['eval_f1']:.3f}",
        "N/A (Generative Task)",
        f"Disc. Acc={disc_acc:.3f} P={disc_precision:.3f} R={disc_recall:.3f} F1={disc_f1:.3f}",
    ],
    "Generative Quality (BLEU/ROUGE/PPL)": [
        "N/A (Classification Task)",
        f"BLEU={bleu_score['bleu']:.4f} ROUGE-L={rouge_score['rougeL']:.4f} PPL={gpt_perplexity:.2f}",
        f"BLEU={gan_bleu['bleu']:.4f} ROUGE-L={gan_rouge['rougeL']:.4f}",
    ],
    "Training Time/Epoch (s)": [f"{bert_time_per_epoch:.1f}", f"{gpt_time_per_epoch:.1f}", f"{gan_time_per_epoch:.1f}"],
})

os.makedirs("results", exist_ok=True)
comparison_df.to_csv("results/comparison_matrix.csv", index=False)
with open("results/gpt2_generation_samples.txt", "w") as f:
    for p, g in zip(sample_prompts, generated_samples):
        f.write(f"PROMPT: {p}\nGENERATED: {g}\n\n")
with open("results/gan_generation_samples.txt", "w") as f:
    for s in gan_samples:
        f.write(s + "\n")

print("Artifacts saved under ./results")
display(comparison_df)

from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

DRIVE_OUT = "/content/drive/MyDrive/nlp_project_outputs"
os.makedirs(DRIVE_OUT, exist_ok=True)

import shutil
for folder in ["results", "splits", "bert_finetuned", "gpt2_finetuned"]:
    if os.path.exists(folder):
        shutil.copytree(folder, os.path.join(DRIVE_OUT, folder), dirs_exist_ok=True)

print(f"Saved to Google Drive at: {DRIVE_OUT}")

Artifacts saved under ./results


,Model Variant,Primary Metric (P/R/F1),Generative Quality (BLEU/ROUGE/PPL),Training Time/Epoch (s)
0,BERT,P=0.986 R=0.996 F1=0.991,N/A (Classification Task),48.6
1,GPT-2 (DistilGPT-2),N/A (Generative Task),BLEU=0.0868 ROUGE-L=0.2349 PPL=13.60,41.8
2,Text-GAN,Disc. Acc=0.988 P=0.976 R=1.000 F1=0.988,BLEU=0.0000 ROUGE-L=0.0096,0.1


Mounted at /content/drive
Saved to Google Drive at: /content/drive/MyDrive/nlp_project_outputs
